In [1]:
import os
import gymnasium
import gymnasium_robotics
from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecVideoRecorder

In [2]:
env_id = 'FetchPushDense-v4'
num_cpu = 4
video_folder = 'logs/videos/'
video_length = 200  # Un episodio de Fetch dura 50 pasos, 200 pasos grabará ~4 episodios
log_dir = "/tmp/gym/"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(video_folder, exist_ok=True)

# 1. Creación del entorno de entrenamiento en paralelo
train_env = make_vec_env(env_id, n_envs=num_cpu)

# 2. Configuración de SAC optimizada para FetchPush
model = SAC(
    policy='MultiInputPolicy',
    env=train_env,
    learning_rate=3e-4,
    buffer_size=500000,      # Reducido un poco para cuidar la memoria RAM
    learning_starts=3000,    # Empezar a aprender un poco antes
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    train_freq=1,
    gradient_steps=1,
    verbose=1,
    seed=42
)

# 3. Entrenamiento (Escala recomendada para éxito)
print("Iniciando entrenamiento...")
model.learn(total_timesteps=500_000)  # Mínimo recomendado para ver éxito consistente
print("Entrenamiento completado.")

# 4. Grabación de video corregida
print("Grabando video de evaluación...")
# SB3 prefiere que el entorno de video se cree a partir de un callable limpio
eval_env = make_vec_env(lambda: gymnasium.make(env_id, render_mode='rgb_array'), n_envs=1)

record_env = VecVideoRecorder(
    eval_env, 
    video_folder,
    record_video_trigger=lambda x: x == 0, 
    video_length=video_length,
    name_prefix=f"sac-{env_id}"
)

# Bucle de ejecución usando la estructura correcta de SB3 VecEnv
obs = record_env.reset()
for _ in range(video_length):
    action, _states = model.predict(obs, deterministic=True)
    # VecEnv siempre devuelve 4 valores: obs, rewards, dones, infos
    obs, rewards, dones, infos = record_env.step(action)

# Cerrar los entornos correctamente para liberar memoria y renderizadores
record_env.close()
eval_env.close()
train_env.close()

print(f"Video guardado en la carpeta: {video_folder}")

Using cpu device
Iniciando entrenamiento...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -10.7    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 289      |
|    time_elapsed    | 0        |
|    total_timesteps | 200      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -8.86    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 8        |
|    fps             | 330      |
|    time_elapsed    | 1        |
|    total_timesteps | 400      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -9.52    |
|    success_rate    | 0        |
| time/              |          |
|   

Moviepy - Done !
Moviepy - video ready c:\Users\Kevin\Desktop\repos\propios\ar2\logs\videos\sac-FetchPushDense-v4-step-0-to-step-200.mp4
Video guardado en la carpeta: logs/videos/
